In [3]:
from operator import matmul
from xml.etree import ElementInclude
import pandas as pd
import numpy as np
from mass_charge_dict import ELEMENTS2Z, Z2ELEMENTS,elements_dict
from scipy import linalg
from math import log10 , floor
import os
import shutil
from functions import *


In [4]:
input_path_coord = 'tests/H2O/coord.xyz'
input_path_hess = 'tests/H2O/hessian'
input_path_dipm = 'tests/H2O/xyz_dipm.csv'

coord,head = import_coord(input_path_coord)
hessian = import_hess(input_path_hess,coord)
dipm = import_dipm(input_path_dipm)

dipm = dipm.iloc[:,:-3]
############
########### Rotation of coordinates and hessian into intermediate position
# Calculating center of mass 
s = center_mass(coord) 
# Translation of coordinate system
vec_trans(coord,s)

#vec_trans(dipm,s)
# Calculating moment of inertia
I = inert_tensor(coord)

# Calculating eigenvalues and eigenvectors 
eig_val,eig_vec = linalg.eigh(I)

# Check if the coordinate system is right-handed --> important for chirality

eig_vec = check_eig_vec(eig_vec)

# Rotating eigenvectors, so that highest values are positive

eig_vec = eig_vec_rot(eig_vec)

# Rotation of the coordinates and atomic dipole moments
coord = coord_rot(coord,eig_vec.copy())

dipm = coord_rot(dipm,eig_vec.copy())

# Construction of the rotation matrix of the hessian and the rotation
P = rotM_hess(eig_vec.copy(),coord)
############

R_euler = get_R_euler(coord,dipm,0,1)

coord_rot(coord,R_euler)
coord_rot(dipm,R_euler)

rotM_Z = rot_Z(10/360*np.pi)

  atoms             x             y         z
0     O -2.941782e-17  5.551115e-17  0.480429
1     H  2.941782e-17 -5.551115e-17 -0.480429
2     H -1.015736e-16 -9.275341e-01  0.731289
   Atom Number        x_dipm    y_dipm    z_dipm
0            8 -1.937435e-18 -0.134911 -0.103270
1            1  4.185415e-18 -0.005167 -0.073520
2            1 -5.298166e-18 -0.072319  0.014207
